# 🧠 Synthetic Human Face Generator (DCGAN 128×128)
Deep Convolutional Generative Adversarial Network trained on the CelebA dataset to synthesize 128×128 photorealistic human faces from 128-dimensional latent noise vectors.

### 1. Setup & GPU Check


In [ ]:
import os, sys, time, random, math
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.utils as vutils
from torch.utils.data import Dataset, DataLoader

if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.backends.cudnn.benchmark = True
    print(f'GPU Active: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('WARNING: Running on CPU! Change runtime to T4 GPU (Runtime -> Change runtime type).')


### 2. Download CelebA Dataset


In [ ]:
!pip install -q kagglehub
import kagglehub

print('Downloading CelebA dataset...')
dataset_path = Path(kagglehub.dataset_download('jessicali9530/celeba-dataset'))

# Locate image directory
candidate_dirs = [
    dataset_path / 'img_align_celeba' / 'img_align_celeba',
    dataset_path / 'img_align_celeba',
    dataset_path,
]
img_dir = next((p for p in candidate_dirs if p.is_dir() and any(p.glob('*.jpg'))), None)
if img_dir is None:
    raise FileNotFoundError(f'Could not find face images inside: {dataset_path}')

print(f'CelebA ready at: {img_dir}')


### 3. Tuned Hyperparameters (Profile A — Balanced & Safe)


In [ ]:
class Config:
    IMAGE_SIZE   = 128       # Resolution: 128x128 RGB
    IMAGE_CHAN   = 3         # 3 color channels
    Z_DIM        = 128       # Latent space dimension
    G_FEATURES   = 64        # Base feature multiplier for G
    D_FEATURES   = 64        # Base feature multiplier for D

    BATCH_SIZE   = 64        # 3,165 updates per epoch
    NUM_EPOCHS   = 20        # Total epochs
    MAX_SAMPLES  = None      # None = full 202,599 images

    LR_G         = 2.0e-4    # Generator learning rate
    LR_D         = 1.5e-4    # TTUR: slightly lower D rate prevents discriminator runaway
    BETA1        = 0.5       # Adam Beta1
    BETA2        = 0.999     # Adam Beta2
    REAL_LABEL   = 0.9       # One-sided label smoothing (prevents D confidence saturation)
    FAKE_LABEL   = 0.0

    USE_COSINE_LR= True      # Cosine LR decay
    ETA_MIN      = 5.0e-5    # LR floor: keeps G learning actively through late epochs
    D_DOMINANCE_THRESH = 0.20 # Extra G update when D(fake) < 0.20
    GRAD_CLIP_G  = 1.0       # Max gradient norm on G

    SEED         = 42
    NUM_WORKERS  = 2
    MIXED_PREC   = True      # FP16 Automatic Mixed Precision
    LOG_INTERVAL = 50        # Telemetry update frequency (batches)

    OUTPUT_DIR   = Path('outputs')
    CKPT_DIR     = Path('outputs/checkpoints')
    SAMPLES_DIR  = Path('outputs/samples')

cfg = Config()
cfg.CKPT_DIR.mkdir(parents=True, exist_ok=True)
cfg.SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)

print(f'DCGAN 128x128 Configured | Batch Size: {cfg.BATCH_SIZE} | Epochs: {cfg.NUM_EPOCHS}')


### 4. Dataset Loader (128×128 Face Cropping)


In [ ]:
class FastCelebADataset(Dataset):
    def __init__(self, img_folder, transform=None, max_samples=None):
        self.img_paths = [
            entry.path for entry in os.scandir(img_folder)
            if entry.name.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        self.img_paths.sort()
        if max_samples and max_samples > 0:
            self.img_paths = self.img_paths[:max_samples]
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        with Image.open(self.img_paths[idx]) as img:
            img = img.convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

transform = T.Compose([
    T.CenterCrop(160),
    T.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE), interpolation=T.InterpolationMode.BILINEAR),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # Scale to [-1, 1]
])

dataset = FastCelebADataset(img_dir, transform=transform, max_samples=cfg.MAX_SAMPLES)
loader = DataLoader(
    dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=(cfg.NUM_WORKERS > 0)
)

print(f'DataLoader Ready: {len(dataset):,} images | {len(loader):,} batches/epoch')


### 5. DCGAN 128×128 Architecture


In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if hasattr(m, 'bias') and m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        if hasattr(m, 'bias') and m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)

class DCGAN_Generator(nn.Module):
    def __init__(self, z_dim=128, ngf=64, nc=3):
        super().__init__()
        self.net = nn.Sequential(
            # z -> 4x4x512
            nn.ConvTranspose2d(z_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # 4x4 -> 8x8x512
            nn.ConvTranspose2d(ngf * 8, ngf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # 8x8 -> 16x16x256
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # 16x16 -> 32x32x128
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # 32x32 -> 64x64x64
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # 64x64 -> 128x128x3
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        if z.dim() == 2:
            z = z.unsqueeze(-1).unsqueeze(-1)
        return self.net(z)

class DCGAN_Discriminator(nn.Module):
    def __init__(self, ndf=64, nc=3):
        super().__init__()
        self.net = nn.Sequential(
            # 128x128 -> 64x64 (No BatchNorm on input layer)
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # 64x64 -> 32x32
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # 32x32 -> 16x16
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # 16x16 -> 8x8
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # 8x8 -> 4x4
            nn.Conv2d(ndf * 8, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # 4x4 -> 1x1 scalar output
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False)
        )

    def forward(self, x):
        return self.net(x).view(-1)

G = DCGAN_Generator(cfg.Z_DIM, cfg.G_FEATURES, cfg.IMAGE_CHAN).to(device)
D = DCGAN_Discriminator(cfg.D_FEATURES, cfg.IMAGE_CHAN).to(device)
G.apply(weights_init)
D.apply(weights_init)

g_params = sum(p.numel() for p in G.parameters()) / 1e6
d_params = sum(p.numel() for p in D.parameters()) / 1e6
print(f'Models Ready: G={g_params:.2f}M | D={d_params:.2f}M | Total={(g_params + d_params):.2f}M params')


### 6. Losses, Optimizers & Schedulers


In [ ]:
criterion = nn.BCEWithLogitsLoss()

opt_G = optim.Adam(G.parameters(), lr=cfg.LR_G, betas=(cfg.BETA1, cfg.BETA2))
opt_D = optim.Adam(D.parameters(), lr=cfg.LR_D, betas=(cfg.BETA1, cfg.BETA2))

if cfg.USE_COSINE_LR:
    sched_G = optim.lr_scheduler.CosineAnnealingLR(opt_G, T_max=cfg.NUM_EPOCHS, eta_min=cfg.ETA_MIN)
    sched_D = optim.lr_scheduler.CosineAnnealingLR(opt_D, T_max=cfg.NUM_EPOCHS, eta_min=cfg.ETA_MIN)
else:
    sched_G, sched_D = None, None

dev_type = 'cuda' if device.type == 'cuda' else 'cpu'
use_amp = (cfg.MIXED_PREC and device.type == 'cuda')

try:
    scaler_G = torch.amp.GradScaler(dev_type, enabled=use_amp)
    scaler_D = torch.amp.GradScaler(dev_type, enabled=use_amp)
except (TypeError, AttributeError):
    scaler_G = torch.cuda.amp.GradScaler(enabled=use_amp)
    scaler_D = torch.cuda.amp.GradScaler(enabled=use_amp)

# Fixed 36 latent noise vectors (6x6 grid for tracking progression)
fixed_noise = torch.randn(36, cfg.Z_DIM, device=device)
print('Optimizers, Schedulers & AMP Scalers ready.')


### 7. Training Loop (128×128)


In [ ]:
def format_time(seconds):
    seconds = int(max(0, seconds))
    h, m, s = seconds // 3600, (seconds % 3600) // 60, seconds % 60
    return f'{h:02d}:{m:02d}:{s:02d}' if h > 0 else f'{m:02d}:{s:02d}'

print(f'Starting DCGAN 128x128 Training: {cfg.NUM_EPOCHS} epochs | Batch Size: {cfg.BATCH_SIZE}')
print('-' * 70)

history = {'epoch': [], 'loss_D': [], 'loss_G': [], 'd_real_prob': [], 'd_fake_prob': [], 'lr': []}
best_equil_score = float('inf')
best_epoch = 0
total_start_time = time.time()
num_batches = len(loader)

for epoch in range(cfg.NUM_EPOCHS):
    epoch_start_time = time.time()
    G.train()
    D.train()

    run_loss_D, run_loss_G = 0.0, 0.0
    run_d_real, run_d_fake = 0.0, 0.0
    batches_done = 0

    for batch_idx, real in enumerate(loader):
        real = real.to(device, non_blocking=True)
        B = real.size(0)

        label_real = torch.full((B,), cfg.REAL_LABEL, dtype=torch.float, device=device)
        label_fake = torch.full((B,), cfg.FAKE_LABEL, dtype=torch.float, device=device)
        label_gen  = torch.full((B,), 1.0, dtype=torch.float, device=device)

        # ── 1. Train Discriminator ─────────────────────────────────────
        z = torch.randn(B, cfg.Z_DIM, device=device)
        with torch.amp.autocast(dev_type, enabled=use_amp):
            out_real = D(real)
            loss_D_real = criterion(out_real, label_real)
            fake = G(z).detach()
            out_fake = D(fake)
            loss_D_fake = criterion(out_fake, label_fake)
            loss_D = loss_D_real + loss_D_fake

        opt_D.zero_grad(set_to_none=True)
        scaler_D.scale(loss_D).backward()
        scaler_D.step(opt_D)
        scaler_D.update()

        # ── 2. Train Generator ─────────────────────────────────────────
        z_gen = torch.randn(B, cfg.Z_DIM, device=device)
        with torch.amp.autocast(dev_type, enabled=use_amp):
            fake_gen = G(z_gen)
            out_fake_gen = D(fake_gen)
            loss_G = criterion(out_fake_gen, label_gen)

        opt_G.zero_grad(set_to_none=True)
        scaler_G.scale(loss_G).backward()
        scaler_G.unscale_(opt_G)
        torch.nn.utils.clip_grad_norm_(G.parameters(), cfg.GRAD_CLIP_G)
        scaler_G.step(opt_G)
        scaler_G.update()

        # ── 3. Adaptive Extra G Step (Counters D Dominance) ────────────
        with torch.no_grad():
            prob_real = torch.sigmoid(out_real).mean().item()
            prob_fake = torch.sigmoid(out_fake).mean().item()

        if prob_fake < cfg.D_DOMINANCE_THRESH:
            z_xtra = torch.randn(B, cfg.Z_DIM, device=device)
            with torch.amp.autocast(dev_type, enabled=use_amp):
                fake_xtra = G(z_xtra)
                loss_G_xtra = criterion(D(fake_xtra), label_gen)
            opt_G.zero_grad(set_to_none=True)
            scaler_G.scale(loss_G_xtra).backward()
            scaler_G.unscale_(opt_G)
            torch.nn.utils.clip_grad_norm_(G.parameters(), cfg.GRAD_CLIP_G)
            scaler_G.step(opt_G)
            scaler_G.update()

        run_loss_D += loss_D.item()
        run_loss_G += loss_G.item()
        run_d_real += prob_real
        run_d_fake += prob_fake
        batches_done += 1

        # In-batch progress update
        if (batch_idx + 1) % cfg.LOG_INTERVAL == 0 or (batch_idx + 1) == num_batches:
            step_time = time.time() - epoch_start_time
            print(
                f'Epoch [{epoch+1:02d}/{cfg.NUM_EPOCHS}] '
                f'[{batch_idx+1:04d}/{num_batches:04d}] | '
                f'Time: {format_time(step_time)} | '
                f'Loss D: {loss_D.item():.4f} | Loss G: {loss_G.item():.4f}',
                end='\r'
            )
            sys.stdout.flush()

    # Step Schedulers
    curr_lr = opt_G.param_groups[0]['lr']
    if sched_G: sched_G.step()
    if sched_D: sched_D.step()

    # End of Epoch: Print time taken and average losses
    epoch_duration = time.time() - epoch_start_time
    avg_loss_D = run_loss_D / batches_done
    avg_loss_G = run_loss_G / batches_done
    avg_prob_real = run_d_real / batches_done
    avg_prob_fake = run_d_fake / batches_done

    history['epoch'].append(epoch + 1)
    history['loss_D'].append(avg_loss_D)
    history['loss_G'].append(avg_loss_G)
    history['d_real_prob'].append(avg_prob_real)
    history['d_fake_prob'].append(avg_prob_fake)
    history['lr'].append(curr_lr)

    print(
        f'\nEpoch [{epoch+1:02d}/{cfg.NUM_EPOCHS}] Finished | '
        f'Time Taken: {format_time(epoch_duration)} | '
        f'Loss D: {avg_loss_D:.4f} | '
        f'Loss G: {avg_loss_G:.4f}'
    )

    # Generate 128x128 Face Grid & Visual Plots
    G.eval()
    with torch.no_grad(), torch.amp.autocast(dev_type, enabled=use_amp):
        samples = G(fixed_noise).float().cpu()
    G.train()

    grid = vutils.make_grid((samples * 0.5 + 0.5).clamp(0, 1), nrow=6, padding=3)
    sample_path = cfg.SAMPLES_DIR / f'epoch_{epoch+1:02d}.png'
    vutils.save_image((samples * 0.5 + 0.5).clamp(0, 1), sample_path, nrow=6, padding=3)

    fig = plt.figure(figsize=(18, 9))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.2, 1])

    ax_img = fig.add_subplot(gs[:, 0])
    ax_img.imshow(grid.permute(1, 2, 0))
    ax_img.axis('off')
    ax_img.set_title(f'Generated Faces (128x128) — Epoch {epoch+1:02d}/{cfg.NUM_EPOCHS}', fontsize=14, fontweight='bold')

    ax_loss = fig.add_subplot(gs[0, 1])
    ax_loss.plot(history['epoch'], history['loss_D'], 'r-o', label='Discriminator Loss', linewidth=2)
    ax_loss.plot(history['epoch'], history['loss_G'], 'b-s', label='Generator Loss', linewidth=2)
    ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('Loss')
    ax_loss.set_title('DCGAN 128x128 Training Losses', fontsize=12, fontweight='bold')
    ax_loss.grid(True, linestyle='--', alpha=0.5)
    ax_loss.legend(loc='upper right')

    ax_prob = fig.add_subplot(gs[1, 1])
    ax_prob.plot(history['epoch'], history['d_real_prob'], 'g-o', label='D(Real) Probability', linewidth=2)
    ax_prob.plot(history['epoch'], history['d_fake_prob'], 'm-s', label='D(Fake) Probability', linewidth=2)
    ax_prob.axhline(0.5, color='gray', linestyle=':', label='Equilibrium (0.5)')
    ax_prob.set_ylim(-0.05, 1.05)
    ax_prob.set_xlabel('Epoch')
    ax_prob.set_ylabel('Probability')
    ax_prob.set_title('Discriminator Discrimination Probabilities', fontsize=12, fontweight='bold')
    ax_prob.grid(True, linestyle='--', alpha=0.5)
    ax_prob.legend(loc='best')

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    # Save latest weights each epoch
    torch.save(G.state_dict(), cfg.CKPT_DIR / 'latest_generator.pth')

    # Save best generator by equilibrium metric
    equil_score = abs(avg_prob_real - 0.5) + abs(avg_prob_fake - 0.5)
    if equil_score < best_equil_score:
        best_equil_score = equil_score
        best_epoch = epoch + 1
        torch.save(G.state_dict(), cfg.CKPT_DIR / 'best_generator.pth')

total_time = time.time() - total_start_time
print('-' * 70)
print(f'Training Complete! Total time: {format_time(total_time)}')
print(f'Best generator saved at: {cfg.CKPT_DIR / "best_generator.pth"} (Epoch {best_epoch:02d})')
print('-' * 70)


### 8. Export Checkpoints & Samples (ZIP Download)


In [ ]:
import zipfile, csv

print('Packaging complete 128x128 training state...')

# 1. Full state bundle (weights, optimizers, history, config)
bundle_path = cfg.CKPT_DIR / 'dcgan128_full_state.pth'
torch.save({
    'epoch'       : len(history['epoch']),
    'resolution'  : cfg.IMAGE_SIZE,
    'G_state'     : G.state_dict(),
    'D_state'     : D.state_dict(),
    'opt_G'       : opt_G.state_dict(),
    'opt_D'       : opt_D.state_dict(),
    'sched_G'     : sched_G.state_dict() if sched_G else None,
    'sched_D'     : sched_D.state_dict() if sched_D else None,
    'fixed_noise' : fixed_noise.cpu(),
    'history'     : history,
    'config'      : {
        'IMAGE_SIZE' : cfg.IMAGE_SIZE,
        'Z_DIM'      : cfg.Z_DIM,
        'G_FEATURES' : cfg.G_FEATURES,
        'D_FEATURES' : cfg.D_FEATURES,
        'LR_G'       : cfg.LR_G,
        'LR_D'       : cfg.LR_D,
        'BETA1'      : cfg.BETA1,
        'BETA2'      : cfg.BETA2,
        'REAL_LABEL' : cfg.REAL_LABEL,
    }
}, bundle_path)
print(f'Full state bundle saved: {bundle_path.stat().st_size / (1024**2):.1f} MB')

# 2. Metrics CSV
metrics_path = cfg.OUTPUT_DIR / 'training_metrics_128.csv'
with open(metrics_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['epoch', 'loss_D', 'loss_G', 'd_real_prob', 'd_fake_prob', 'lr'])
    w.writeheader()
    for i in range(len(history['epoch'])):
        w.writerow({
            'epoch'       : history['epoch'][i],
            'loss_D'      : round(history['loss_D'][i], 6),
            'loss_G'      : round(history['loss_G'][i], 6),
            'd_real_prob' : round(history['d_real_prob'][i], 6),
            'd_fake_prob' : round(history['d_fake_prob'][i], 6),
            'lr'          : history['lr'][i],
        })
print(f'Metrics CSV saved: {metrics_path}')

# 3. ZIP package
zip_path = Path('/content/dcgan128_export.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(bundle_path, 'dcgan128_full_state.pth')
    zf.write(metrics_path, 'training_metrics_128.csv')
    for name in ['best_generator.pth', 'latest_generator.pth']:
        p = cfg.CKPT_DIR / name
        if p.exists(): zf.write(p, name)
    for p in sorted(cfg.SAMPLES_DIR.glob('epoch_*.png')):
        zf.write(p, f'samples/{p.name}')

print(f'ZIP package created: {zip_path.stat().st_size / (1024**2):.1f} MB')

# 4. Trigger browser download
try:
    from google.colab import files
    print('Triggering browser download...')
    files.download(str(zip_path))
except ImportError:
    print(f'Non-Colab environment. File saved at: {zip_path.resolve()}')
